In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Démarrage rapide avec Sampler

La tâche principale de Sampler est d'échantillonner le registre de sortie à partir de l'exécution d'un ou plusieurs circuits quantiques. Les [circuits dynamiques](/guides/execute-dynamic-circuits) et les circuits paramétrés sont acceptés en entrée (si des circuits paramétrés sont soumis, les valeurs des paramètres doivent également être fournies). Sampler prend également en charge le découplage dynamique intégré et la torsion pour la [suppression d'erreurs](/guides/error-mitigation-and-suppression-techniques).

Les étapes de ce sujet décrivent comment configurer Sampler, explorer les options que tu peux utiliser pour le configurer et l'invoquer dans un programme.


<Accordion>
<AccordionItem title="Versions des packages">

Le code sur cette page a été développé en utilisant les exigences suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

## Étapes pour utiliser la primitive Sampler
### 1. Initialiser le compte
Parce que Qiskit Runtime est un service géré, tu dois d'abord initialiser ton compte. Tu peux ensuite sélectionner la QPU que tu veux utiliser pour calculer la valeur d'espérance.

Suis les étapes de la rubrique [Configurer ton compte IBM Cloud](/guides/cloud-setup) si tu n'as pas encore configuré de compte.

:::note[Gates fractionnaires]

Pour utiliser les [gates fractionnaires](/guides/fractional-gates) récemment pris en charge, définis `use_fractional_gates=True` lors de la demande d'un Backend à partir d'une instance `QiskitRuntimeService`. Par exemple :

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

Il s'agit d'une fonctionnalité expérimentale qui pourrait changer à l'avenir.

:::

In [2]:
import numpy as np
from qiskit.circuit.library import efficient_su2

circuit = efficient_su2(127, entanglement="linear")
circuit.measure_all()
# The circuit is parametrized, so we will define the parameter values for execution
param_values = np.random.rand(circuit.num_parameters)

### 2. Créer un circuit
Tu as besoin d'au moins un circuit comme entrée pour la primitive Sampler.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 3036), ('sx', 1769), ('cz', 378), ('measure', 127), ('barrier', 1)])


Le circuit et l'observable doivent être transformés pour n'utiliser que des instructions prises en charge par la QPU (appelées circuits *d'architecture d'ensemble d'instructions (ISA)*). Utilise le Transpiler pour ce faire.

In [4]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

### 4. Invoke Sampler and get results

Next, invoke the `run()` method to generate the output. The circuit and optional parameter value sets are input as *primitive unified bloc* (PUB) tuples.

In [5]:
job = sampler.run([(isa_circuit, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d82863mgbeec73alf9sg


>>> Job Status: QUEUED


In [ ]:
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]
print(
    f"First ten results for the 'meas' output register: "
    f"{pub_result.data.meas.get_bitstrings()[:10]}"
)

First ten results for the 'meas' output register: ['1100110011001011111111111010000010001010100100011000001011001101000110011000110100100100101010111001110100100000000011111100000', '0101001001010000100111000110110001001101010110110000110111101110001100000001000001111111101110000000010011111100100110001101000', '0111111110011011000011110111010111101100110010001010010001100000000100000000001010101010111010110000001100100001010110000101000', '0000110011001100110011101100000111011001110100001100001100110111010100101010001010000011000111001010101111110110100110001010000', '0011110011100001100110111001000011011111011110111100000110001000111011101101000110011011101011001110110000010010001100100011001', '1010001000010101011100101010101001101000100010011011100110010111010001110111110010100010111010011010110011001101100110010000010', '0001110010001011001100010000000001001101001110101100110011101111100100100110110010101000011010101000101011101011010100000101010', '1110100100001100110010000010011

### 4. Invoquer Sampler et obtenir les résultats
Ensuite, invoque la méthode `run()` pour générer la sortie. Le circuit et les ensembles de valeurs de paramètres optionnels sont entrés sous forme de tuples *bloc unifié de primitive* (PUB).